In [ ]:
import os
import time
import joblib
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from lightgbm import LGBMClassifier

# OPTUNA
import optuna

In [ ]:
# ==========================================
# 1. PREPARACIÓN DE DATOS (POLARS)
# ==========================================

path_train = "../DATASETS/dataSets_Reducidos/nusw-nb15/datos_train_NUSW_redux.csv"
path_test  = "../DATASETS/dataSets_Reducidos/nusw-nb15/datos_test_NUSW_redux.csv"

df_train = pl.read_csv(path_train)
df_test  = pl.read_csv(path_test)

TARGET_COL = "attack_cat"

# LightGBM prefiere etiquetas 0 (Ataque) y 1 (Normal)
y_full_train = (
    df_train.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal").then(1).otherwise(0).alias("label")
    ).to_series().to_numpy()
)

y_test_np = (
    df_test.select(
        pl.when(pl.col(TARGET_COL).str.strip_chars() == "Normal").then(1).otherwise(0).alias("label")
    ).to_series().to_numpy()
)

X_full_train = df_train.drop(TARGET_COL).to_numpy()
X_test_np = df_test.drop(TARGET_COL).to_numpy()

print(f"✅ Datos listos. Train: {X_full_train.shape[0]} muestras, Test: {X_test_np.shape[0]} muestras.")

In [ ]:
# ==========================================
# 2. FASE 1: OPTUNA + 3-FOLD CV (ESTRUCTURA REAL)
# ==========================================

def objective(trial):
    # Sugerimos parámetros que realmente definen la arquitectura en LGBM
    n_estimators = trial.suggest_int("n_estimators", 50, 600, step=50)
    num_leaves = trial.suggest_int("num_leaves", 15, 255) # Control real de complejidad
    max_depth = trial.suggest_int("max_depth", 3, 12)    # Límite para evitar overfitting
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_scores = []
    latencies = []

    for train_idx, val_idx in skf.split(X_full_train, y_full_train):
        X_train_cv, X_val_cv = X_full_train[train_idx], X_full_train[val_idx]
        y_train_cv, y_val_cv = y_full_train[train_idx], y_full_train[val_idx]

        model = LGBMClassifier(
            n_estimators=n_estimators,
            num_leaves=num_leaves,
            max_depth=max_depth,
            learning_rate=0.1,  # Baseline estable
            n_jobs=-1,          # Máximo uso de CPU
            random_state=42,
            verbosity=-1        # Evitar spam en consola
        )

        model.fit(X_train_cv, y_train_cv)

        # 1. Medir Eficacia
        y_pred = model.predict(X_val_cv)
        f1_scores.append(f1_score(y_val_cv, y_pred, average="macro"))

        # 2. Medir Eficiencia (5 repeticiones para mitigar ruido)
        subset = min(20000, len(X_val_cv))
        X_lat = X_val_cv[:subset]
        
        # Warm-up rápido
        _ = model.predict(X_lat[:500])
        
        rep_lat = []
        for _ in range(5):
            t0 = time.perf_counter()
            _ = model.predict(X_lat)
            t1 = time.perf_counter()
            rep_lat.append((t1 - t0) / len(X_lat) * 1000)
        
        latencies.append(np.mean(rep_lat))

    avg_f1 = float(np.mean(f1_scores))
    avg_lat = float(np.mean(latencies))
    trial.set_user_attr("f1_std", float(np.std(f1_scores)))

    return avg_f1, avg_lat

# Ejecución del estudio
study = optuna.create_study(directions=["maximize", "minimize"], study_name="lgbm_ids_optimization")
print("🚀 Iniciando barrido multiobjetivo con LightGBM...")
study.optimize(objective, n_trials=50)

# ==========================================
# 3. EXPORTACIÓN Y VISUALIZACIÓN PARETO
# ==========================================

pareto_ids = {t.number for t in study.best_trials}
trials_data = []
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        trials_data.append({
            "n_estimators": t.params["n_estimators"],
            "num_leaves": t.params["num_leaves"],
            "max_depth": t.params["max_depth"],
            "f1_macro": t.values[0],
            "f1_std": t.user_attrs["f1_std"],
            "latency_ms": t.values[1],
            "is_pareto": t.number in pareto_ids
        })

df_results = pl.DataFrame(trials_data)
df_results.write_csv("lgbm_trials_results_cv.csv")

In [ ]:
# Gráfico usando NumPy bridge para evitar dependencia de PyArrow/Pandas
plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")
sns.scatterplot(
    x=df_results["latency_ms"].to_numpy(),
    y=df_results["f1_macro"].to_numpy(),
    hue=df_results["is_pareto"].to_numpy(),
    palette={True: "red", False: "royalblue"},
    style=df_results["is_pareto"].to_numpy(),
    markers={True: "D", False: "o"}, s=100, alpha=0.8
)

# Anotación de puntos Pareto
pareto_points = df_results.filter(pl.col("is_pareto") == True)
for row in pareto_points.iter_rows(named=True):
    plt.text(row["latency_ms"], row["f1_macro"] + 0.0005, 
             f"n={int(row['n_estimators'])}, l={int(row['num_leaves'])}", 
             fontsize=8, fontweight='bold', ha='center')

plt.title("Frontera de Pareto: LightGBM (Eficiencia vs Eficacia)")
plt.xlabel("Latencia Media (ms/muestra)")
plt.ylabel("F1-Score Macro")
plt.show()

In [ ]:
# ==========================================
# 4. FASE 3: EVALUACIÓN FINAL EN TEST
# ==========================================

# Aquí deberás ajustar estos candidatos con los que veas en ROJO en tu gráfica
candidatos = [
    {"n": 100, "l": 31, "d": 5,  "nombre": "LGBM - Alta Eficiencia"},
    {"n": 200, "l": 63, "d": 8,  "nombre": "LGBM - Equilibrado"},
    {"n": 500, "l": 127, "d": 12, "nombre": "LGBM - Máxima Precisión"}
]

resultados_finales = []

print("\n--- EVALUACIÓN FINAL SOBRE TEST (COHERENTE) ---")
for c in candidatos:
    model_final = LGBMClassifier(
        n_estimators=c["n"],
        num_leaves=c["l"],
        max_depth=c["d"],
        learning_rate=0.1, # Coherencia con la Fase 1
        n_jobs=-1,
        random_state=42,
        verbosity=-1
    )
    
    model_final.fit(X_full_train, y_full_train)
    
    # Medición de latencia real en Test (5 pasadas)
    t_start = time.perf_counter()
    y_test_pred = model_final.predict(X_test_np)
    t_end = time.perf_counter()
    
    resultados_finales.append({
        "Perfil": c["nombre"],
        "n_est": c["n"],
        "leaves": c["l"],
        "F1_Test": f1_score(y_test_np, y_test_pred, average="macro"),
        "Acc_Test": accuracy_score(y_test_np, y_test_pred),
        "Latencia_ms": ((t_end - t_start) / len(X_test_np)) * 1000
    })

df_final = pl.DataFrame(resultados_finales)
print("\n", df_final)